In [ ]:
# 1: Load dataset and verify structure

import pandas as pd

PATH = "whl_2025_ordered.csv"

df = pd.read_csv(PATH)

# show basic info
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values per column:")
print(df.isna().sum())

print("\nBasic stats for TOI:")
print(df["toi"].describe())

In [ ]:
# 2: Compute league BaseRate (xG per second)

# Total xG in each segment (both teams)
total_xg = (df["home_xg"] + df["away_xg"]).sum()

# Total time across all segments (seconds)
total_toi = df["toi"].sum()

base_rate = total_xg / total_toi  # xG per second

print("Total xG (home + away):", total_xg)
print("Total TOI (seconds):", total_toi)
print("BaseRate (xG per second):", base_rate)

# League xG per 60 per TEAM ≈ (base_rate * 60*60) / 2  (divide by 2 because base_rate included both teams)
league_xg60_per_team = (base_rate * 3600) / 2
print("League average xG per 60 per team:", league_xg60_per_team)


In [ ]:
# STEP 3 (weighted): Offensive line effects by TEAM + line label (even strength only)

valid_lines = ["first_off", "second_off"]
df_even = df[df["home_off_line"].isin(valid_lines)].copy()

league_rate_per_team = base_rate / 2

# Weighted xG rate = sum(xG) / sum(TOI) for each (team, line)
team_line_weighted = (
    df_even.groupby(["home_team", "home_off_line"])
      .agg(line_xg_sum=("home_xg", "sum"),
           line_toi_sum=("toi", "sum"))
      .reset_index()
)

team_line_weighted["line_xg_rate"] = team_line_weighted["line_xg_sum"] / team_line_weighted["line_toi_sum"]
team_line_weighted["off_line_effect"] = team_line_weighted["line_xg_rate"] - league_rate_per_team

# Sort for inspection
team_line_weighted_sorted = team_line_weighted.sort_values("off_line_effect", ascending=False).reset_index(drop=True)

print("Top 15 (team, line) offensive effects - WEIGHTED:")
display(team_line_weighted_sorted.head(15))

print("\nBottom 15 (team, line) offensive effects - WEIGHTED:")
display(team_line_weighted_sorted.tail(15))


In [ ]:
# STEP 3B (weighted): clean even-strength offensive table (first_off & second_off only)

team_line_even = team_line_weighted.sort_values(
    ["home_team", "home_off_line"]
).reset_index(drop=True)

print("Even-strength offensive effects (weighted):")
display(team_line_even.head(20))


In [ ]:
# STEP 4 (weighted): Defensive pairing effects by TEAM + pairing label (even strength only)

valid_lines = ["first_off", "second_off"]
df_even = df[df["home_off_line"].isin(valid_lines)].copy()

league_rate_per_team = base_rate / 2

# Away xG rate represents what the HOME defense allowed in that segment
# Weighted allowed rate = sum(away_xg) / sum(TOI)
team_def_weighted = (
    df_even.groupby(["home_team", "home_def_pairing"])
      .agg(allowed_xg_sum=("away_xg", "sum"),
           def_toi_sum=("toi", "sum"))
      .reset_index()
)

team_def_weighted["xg_allowed_rate"] = team_def_weighted["allowed_xg_sum"] / team_def_weighted["def_toi_sum"]

# Defensive effect: positive = allows less than league avg
team_def_weighted["def_pair_effect"] = league_rate_per_team - team_def_weighted["xg_allowed_rate"]

team_def_weighted_sorted = team_def_weighted.sort_values("def_pair_effect", ascending=False).reset_index(drop=True)

print("Top 15 strongest defensive pairings - WEIGHTED:")
display(team_def_weighted_sorted.head(15))

print("\nBottom 15 weakest defensive pairings - WEIGHTED:")
display(team_def_weighted_sorted.tail(15))


In [ ]:
# STEP 4.5 (CORRECTED): Goalie effect using goals vs expected goals (5v5-only)

valid_lines = ["first_off", "second_off"]
df_even = df[df["home_off_line"].isin(valid_lines)].copy()

# Home goalie faces away shots
home_goalie_tbl = (
    df_even.groupby("home_goalie")
      .agg(
          xg_faced=("away_xg", "sum"),
          goals_allowed=("away_goals", "sum"),
          toi_sum=("toi", "sum")
      )
      .reset_index()
      .rename(columns={"home_goalie": "goalie"})
)

# Away goalie faces home shots
away_goalie_tbl = (
    df_even.groupby("away_goalie")
      .agg(
          xg_faced=("home_xg", "sum"),
          goals_allowed=("home_goals", "sum"),
          toi_sum=("toi", "sum")
      )
      .reset_index()
      .rename(columns={"away_goalie": "goalie"})
)

goalie_weighted = pd.concat([home_goalie_tbl, away_goalie_tbl], ignore_index=True)

goalie_weighted = (
    goalie_weighted.groupby("goalie", as_index=False)
      .agg(
          xg_faced=("xg_faced", "sum"),
          goals_allowed=("goals_allowed", "sum"),
          toi_sum=("toi_sum", "sum")
      )
)

# Rates per second
goalie_weighted["xg_allowed_rate"] = goalie_weighted["xg_faced"] / goalie_weighted["toi_sum"]
goalie_weighted["goals_allowed_rate"] = goalie_weighted["goals_allowed"] / goalie_weighted["toi_sum"]

# TRUE goalie effect
goalie_weighted["goalie_effect"] = (
    goalie_weighted["xg_allowed_rate"] - goalie_weighted["goals_allowed_rate"]
)

goalie_weighted = goalie_weighted.sort_values("goalie_effect", ascending=False).reset_index(drop=True)

print("Top goalies (save more than expected):")
display(goalie_weighted.head(10))

print("\nWorst goalies (allow more than expected):")
display(goalie_weighted.tail(10))


In [ ]:
# STEP 5: Team RAPM ratings (5v5-only), TOI-weighted, INCLUDING goalie effect
# Requires these to already exist from earlier steps:
# - base_rate
# - team_line_weighted  (from weighted Step 3)
# - team_def_weighted   (from weighted Step 4)
# - goalie_weighted     (from Step 4.5)
# And df loaded from Step 1.

import pandas as pd

# --- Filter to even-strength segments (keep consistent everywhere) ---
valid_lines = ["first_off", "second_off"]
df_even = df[df["home_off_line"].isin(valid_lines)].copy()

# --- TEAM OFFENSIVE EFFECT (TOI-weighted across first_off & second_off) ---
team_off = team_line_weighted.copy()
team_off["weighted_off"] = team_off["off_line_effect"] * team_off["line_toi_sum"]

team_off = (
    team_off.groupby("home_team", as_index=False)
            .agg(weighted_off=("weighted_off", "sum"),
                 toi_sum=("line_toi_sum", "sum"))
)

team_off["team_off_effect"] = team_off["weighted_off"] / team_off["toi_sum"]
team_off = team_off[["home_team", "team_off_effect"]]

# --- TEAM DEFENSIVE EFFECT (TOI-weighted across defensive pairings) ---
team_def = team_def_weighted.copy()
team_def["weighted_def"] = team_def["def_pair_effect"] * team_def["def_toi_sum"]

team_def = (
    team_def.groupby("home_team", as_index=False)
            .agg(weighted_def=("weighted_def", "sum"),
                 toi_sum=("def_toi_sum", "sum"))
)

team_def["team_def_effect"] = team_def["weighted_def"] / team_def["toi_sum"]
team_def = team_def[["home_team", "team_def_effect"]]

# --- TEAM GOALIE EFFECT (TOI-weighted across goalies used by the team) ---

# Home goalie TOI belongs to home_team
team_goalie_home = (
    df_even.groupby(["home_team", "home_goalie"], as_index=False)
          .agg(goalie_toi=("toi", "sum"))
          .rename(columns={"home_team": "team", "home_goalie": "goalie"})
)

# Away goalie TOI belongs to away_team
team_goalie_away = (
    df_even.groupby(["away_team", "away_goalie"], as_index=False)
          .agg(goalie_toi=("toi", "sum"))
          .rename(columns={"away_team": "team", "away_goalie": "goalie"})
)

team_goalie_toi = pd.concat([team_goalie_home, team_goalie_away], ignore_index=True)

# Attach each goalie's goalie_effect
team_goalie_toi = team_goalie_toi.merge(
    goalie_weighted[["goalie", "goalie_effect"]],
    on="goalie",
    how="left"
)

# If any goalie_effect is missing (should be rare), fill with 0 (league average)
team_goalie_toi["goalie_effect"] = team_goalie_toi["goalie_effect"].fillna(0.0)

# Weighted average goalie effect per team
team_goalie_toi["weighted_goalie"] = team_goalie_toi["goalie_effect"] * team_goalie_toi["goalie_toi"]

team_goalie = (
    team_goalie_toi.groupby("team", as_index=False)
                  .agg(weighted_goalie=("weighted_goalie", "sum"),
                       toi_sum=("goalie_toi", "sum"))
)

team_goalie["team_goalie_effect"] = team_goalie["weighted_goalie"] / team_goalie["toi_sum"]
team_goalie = team_goalie.rename(columns={"team": "home_team"})[["home_team", "team_goalie_effect"]]

# --- COMBINE INTO NET TEAM RAPM ---
team_rapm = team_off.merge(team_def, on="home_team", how="inner")
team_rapm = team_rapm.merge(team_goalie, on="home_team", how="left")

team_rapm["team_net_rapm"] = (
    team_rapm["team_off_effect"]
    + team_rapm["team_def_effect"]
    + team_rapm["team_goalie_effect"]
)

# Sort best to worst and add rank
team_rapm = team_rapm.sort_values("team_net_rapm", ascending=False).reset_index(drop=True)
team_rapm.insert(0, "rank", team_rapm.index + 1)

print("TEAM POWER RANKINGS (RAPM-style, 5v5-only, includes goalie):")
display(team_rapm)


In [ ]:
# STEP 6: Regression RAPM (5v5-only), symmetric rows, Ridge regularization

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

valid_lines = ["first_off", "second_off"]
df_even = df[df["home_off_line"].isin(valid_lines)].copy()

# Avoid divide-by-zero (shouldn't happen, but safe)
df_even = df_even[df_even["toi"] > 0].copy()

# Build TWO rows per segment so home/away are treated symmetrically:
# Row type 1: home offense vs away defense/goalie
home_rows = pd.DataFrame({
    "off_team": df_even["home_team"],
    "off_line": df_even["home_off_line"],
    "def_team": df_even["away_team"],
    "def_pair": df_even["away_def_pairing"],
    "goalie": df_even["away_goalie"],
    "is_home_off": 1,
    "xg": df_even["home_xg"],
    "toi": df_even["toi"],
})

# Row type 2: away offense vs home defense/goalie
away_rows = pd.DataFrame({
    "off_team": df_even["away_team"],
    "off_line": df_even["away_off_line"],
    "def_team": df_even["home_team"],
    "def_pair": df_even["home_def_pairing"],
    "goalie": df_even["home_goalie"],
    "is_home_off": 0,
    "xg": df_even["away_xg"],
    "toi": df_even["toi"],
})

rapm_rows = pd.concat([home_rows, away_rows], ignore_index=True)

# Target: xG rate (per second)
rapm_rows["xg_rate"] = rapm_rows["xg"] / rapm_rows["toi"]

# Feature construction:
# Combine team+unit labels so "first_off" is team-specific (important!)
rapm_rows["off_unit"] = rapm_rows["off_team"].astype(str) + "__" + rapm_rows["off_line"].astype(str)
rapm_rows["def_unit"] = rapm_rows["def_team"].astype(str) + "__" + rapm_rows["def_pair"].astype(str)

# One-hot encode categorical features
X = pd.get_dummies(
    rapm_rows[["off_unit", "def_unit", "goalie", "is_home_off"]],
    drop_first=False
)

y = rapm_rows["xg_rate"].values

# Use TOI as sample weight so longer segments matter more
w = rapm_rows["toi"].values

# Ridge regularization = shrinkage + stability (this is the RAPM "R")
# alpha controls shrinkage strength (bigger = more shrinkage)
model = Ridge(alpha=50.0, fit_intercept=True, random_state=0)
model.fit(X, y, sample_weight=w)

# Store coefficients
coef = pd.Series(model.coef_, index=X.columns)
intercept = model.intercept_

print("Ridge RAPM fitted. Intercept (baseline xG_rate):", intercept)
print("Home-ice coefficient (is_home_off):", coef.get("is_home_off", np.nan))


In [ ]:
# STEP 7: Build team power rankings from regression RAPM coefficients (xG-based)

# Helper: get coefficient for a named feature (0 if missing)
def get_coef(name: str) -> float:
    return float(coef[name]) if name in coef.index else 0.0

teams = sorted(pd.unique(pd.concat([df["home_team"], df["away_team"]], ignore_index=True)))

# Weights: use actual TOI usage for each team's units (so first line matters more if it plays more)
# Build TOI summaries for offense units and defense units using the symmetric rapm_rows

off_usage = (
    rapm_rows.groupby(["off_team", "off_line"], as_index=False)
             .agg(toi=("toi", "sum"))
)
def_usage = (
    rapm_rows.groupby(["def_team", "def_pair"], as_index=False)
             .agg(toi=("toi", "sum"))
)

rows = []
for t in teams:
    # Offensive rating: TOI-weighted mean of team’s first/second offensive unit coefficients
    off_t = off_usage[off_usage["off_team"] == t].copy()
    off_t["unit_name"] = off_t["off_team"].astype(str) + "__" + off_t["off_line"].astype(str)
    off_t["unit_coef"] = off_t["unit_name"].map(lambda u: get_coef(f"off_unit_{u}"))
    team_off = (off_t["unit_coef"] * off_t["toi"]).sum() / off_t["toi"].sum()

    # Defensive rating: defense suppresses xG, but the regression coefficient on def_unit
    # is "how def_unit changes opponent xG rate". Lower is better.
    # So we flip sign so higher = better defense.
    def_t = def_usage[def_usage["def_team"] == t].copy()
    def_t["unit_name"] = def_t["def_team"].astype(str) + "__" + def_t["def_pair"].astype(str)
    def_t["unit_coef_raw"] = def_t["unit_name"].map(lambda u: get_coef(f"def_unit_{u}"))
    team_def = - (def_t["unit_coef_raw"] * def_t["toi"]).sum() / def_t["toi"].sum()

    rows.append({"team": t, "off_rapm": team_off, "def_rapm": team_def, "net_rapm_xg": team_off + team_def})

team_rank_xg = pd.DataFrame(rows).sort_values("net_rapm_xg", ascending=False).reset_index(drop=True)
team_rank_xg.insert(0, "rank", team_rank_xg.index + 1)

display(team_rank_xg.head(32))

# Convert RAPM from per-second to per-60-minute units (standard hockey units)

team_rank_xg["off_rapm_per60"] = team_rank_xg["off_rapm"] * 3600
team_rank_xg["def_rapm_per60"] = team_rank_xg["def_rapm"] * 3600
team_rank_xg["net_rapm_per60"] = team_rank_xg["net_rapm_xg"] * 3600

# Re-sort using per60 units (same order, just clearer units)
team_rank_xg = team_rank_xg.sort_values("net_rapm_per60", ascending=False).reset_index(drop=True)
team_rank_xg["rank"] = team_rank_xg.index + 1

print("TEAM POWER RANKINGS (xG RAPM, per 60 minutes):")
display(team_rank_xg[["rank", "team", "off_rapm_per60", "def_rapm_per60", "net_rapm_per60"]])



In [ ]:
# STEP 8: Goalie shot-stopping effect (GSAx-style rate), then team average

df_even = df[df["home_off_line"].isin(valid_lines)].copy()
df_even = df_even[df_even["toi"] > 0].copy()

home_goalie_tbl = (
    df_even.groupby("home_goalie", as_index=False)
          .agg(xg_faced=("away_xg", "sum"),
               goals_allowed=("away_goals", "sum"),
               toi=("toi", "sum"))
          .rename(columns={"home_goalie": "goalie"})
)

away_goalie_tbl = (
    df_even.groupby("away_goalie", as_index=False)
          .agg(xg_faced=("home_xg", "sum"),
               goals_allowed=("home_goals", "sum"),
               toi=("toi", "sum"))
          .rename(columns={"away_goalie": "goalie"})
)

goalies = pd.concat([home_goalie_tbl, away_goalie_tbl], ignore_index=True)
goalies = goalies.groupby("goalie", as_index=False).sum()

goalies["xg_rate"] = goalies["xg_faced"] / goalies["toi"]
goalies["ga_rate"] = goalies["goals_allowed"] / goalies["toi"]
goalies["goalie_gsax_rate"] = goalies["xg_rate"] - goalies["ga_rate"]  # + = saves more than expected

# Team goalie usage (TOI-weighted across goalies used by each team, symmetric)
team_goalie_home = (
    df_even.groupby(["home_team", "home_goalie"], as_index=False)
          .agg(toi=("toi", "sum"))
          .rename(columns={"home_team": "team", "home_goalie": "goalie"})
)
team_goalie_away = (
    df_even.groupby(["away_team", "away_goalie"], as_index=False)
          .agg(toi=("toi", "sum"))
          .rename(columns={"away_team": "team", "away_goalie": "goalie"})
)

team_goalie_use = pd.concat([team_goalie_home, team_goalie_away], ignore_index=True)
team_goalie_use = team_goalie_use.merge(goalies[["goalie", "goalie_gsax_rate"]], on="goalie", how="left")
team_goalie_use["goalie_gsax_rate"] = team_goalie_use["goalie_gsax_rate"].fillna(0.0)

team_goalie_use["weighted"] = team_goalie_use["goalie_gsax_rate"] * team_goalie_use["toi"]
team_goalie = (
    team_goalie_use.groupby("team", as_index=False)
                   .agg(weighted=("weighted", "sum"), toi=("toi", "sum"))
)
team_goalie["team_goalie_gsax_rate"] = team_goalie["weighted"] / team_goalie["toi"]
# Convert goalie gsax to per-60-minute units
team_goalie["team_goalie_gsax_per60"] = team_goalie["team_goalie_gsax_rate"] * 3600


display(team_goalie.sort_values("team_goalie_gsax_rate", ascending=False).head(10))


In [ ]:
# STEP 9: Final power rating = xG-RAPM (per60) + goalie adjustment (per60)

LAMBDA = 0.35

# Convert goalie gsax to per60
team_goalie = team_goalie.copy()
team_goalie["team_goalie_gsax_per60"] = team_goalie["team_goalie_gsax_rate"] * 3600

# Merge goalie data
final_rank = team_rank_xg.merge(
    team_goalie[["team", "team_goalie_gsax_per60"]],
    on="team",
    how="left"
)

final_rank["team_goalie_gsax_per60"] = final_rank["team_goalie_gsax_per60"].fillna(0.0)

# Final blended rating
final_rank["final_power_per60"] = (
    final_rank["net_rapm_per60"]
    + LAMBDA * final_rank["team_goalie_gsax_per60"]
)

# Rank teams
final_rank = final_rank.sort_values("final_power_per60", ascending=False).reset_index(drop=True)
final_rank["rank"] = final_rank.index + 1

# DISPLAY INCLUDING OFFENSE AND DEFENSE
display(final_rank[[
    "rank",
    "team",
    "off_rapm_per60",
    "def_rapm_per60",
    "net_rapm_per60",
    "team_goalie_gsax_per60",
    "final_power_per60"
]])


In [ ]:
# Export power rankings to CSV
final_rank.to_csv("power_rankings_whl_2025.csv", index=False)

print("CSV file saved as power_rankings_whl_2025.csv")


In [ ]:
# STEP 10: Team finishing diagnostics (Goals - xG)

# Build per-team totals from full game logs (not just 5v5 if you prefer, but here: 5v5 only)
df_even = df[df["home_off_line"].isin(valid_lines)].copy()

# Goals/xG for each team in symmetric way
home_team_tot = df_even.groupby("home_team", as_index=False).agg(
    xg_for=("home_xg", "sum"),
    g_for=("home_goals", "sum"),
    xg_against=("away_xg", "sum"),
    g_against=("away_goals", "sum"),
    toi=("toi", "sum"),
).rename(columns={"home_team": "team"})

away_team_tot = df_even.groupby("away_team", as_index=False).agg(
    xg_for=("away_xg", "sum"),
    g_for=("away_goals", "sum"),
    xg_against=("home_xg", "sum"),
    g_against=("home_goals", "sum"),
    toi=("toi", "sum"),
).rename(columns={"away_team": "team"})

tot = pd.concat([home_team_tot, away_team_tot], ignore_index=True).groupby("team", as_index=False).sum()

tot["finishing_for"] = tot["g_for"] - tot["xg_for"]                # + = scoring more than xG
tot["finishing_against"] = tot["xg_against"] - tot["g_against"]    # + = allowing fewer goals than xG
tot["finishing_net"] = tot["finishing_for"] + tot["finishing_against"]

display(tot.sort_values("finishing_net", ascending=False).head(10))
display(tot.sort_values("finishing_net", ascending=True).head(10))
